<a href="https://colab.research.google.com/github/sergiocalazans/big-data-pyspark/blob/main/RDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
#importar bibliotecas
import pyspark
from pyspark.sql import SparkSession
from pyspark import SparkFiles

In [62]:
#criar sessão e contexto do Spark
spark = SparkSession.builder.appName('ex1').getOrCreate()

#criar o contexto
sc = spark.sparkContext

In [63]:
#criando RDD a partir de uma lista com parallelize
lista = [1,2,3,4,5,6]

rdd = sc.parallelize(lista)#transformação

rdd.collect()#operação

[1, 2, 3, 4, 5, 6]

In [64]:
type(rdd)

pyspark.core.rdd.RDD

In [65]:
#Transformações ---MAP ---x2
rdd2 = rdd.\
  map(lambda x: x*2)
rdd2.take(2)

[2, 4]

In [66]:
rdd2.collect()

[2, 4, 6, 8, 10, 12]

In [67]:
#Transformações ---FLAPMAP -- x --> [x,x*2]
rdd2.\
  flatMap(lambda x: [x, x*2]).\
  collect()

[2, 4, 4, 8, 6, 12, 8, 16, 10, 20, 12, 24]

In [68]:
rdd.collect()

[1, 2, 3, 4, 5, 6]

In [69]:
#Transformações ---FILTER
rdd.\
  filter(lambda x: x%2==1).\
  take(10)

[1, 3, 5]

In [70]:
#Transformações --union-DISTINCT
rdd2.\
  union(rdd).\
  filter(lambda x: x%2==1).\
  distinct().\
  take(12)



[1, 5, 3]

In [71]:
#Transformações ---INTERSECTION
rdd.\
  intersection(rdd2).\
  take(12)

[4, 2, 6]

In [72]:
rdd.collect()

[1, 2, 3, 4, 5, 6]

In [73]:
#Transformações ---SUBTRACT
rdd.\
  subtract(rdd2).collect()

[1, 5, 3]

In [74]:
######################
## Actions ou Operations

In [75]:
# Action --- COLLECT
rdd.collect()

[1, 2, 3, 4, 5, 6]

In [76]:
# Action --- TAKE
rdd2.take(3)

[2, 4, 6]

In [77]:
# Action --- COUNT
rdd2.count()


6

In [78]:
# Action --- COUNT by value
rdd.union(rdd2).\
  countByValue()

defaultdict(int, {1: 1, 2: 2, 3: 1, 4: 2, 5: 1, 6: 2, 8: 1, 10: 1, 12: 1})

In [79]:
rdd.union(rdd2).\
  reduce(lambda x, y: x+y)


63

In [80]:
#######################################################
#Base Euro 2024
#1. transformar o nome dos jogadores em letra maiúscula
#2. Exibir todos os jogadores da Alemanha
#3. Contar número de jogadores da Inglaterra com idade menor que 25 anos
#4. Listar jogadores da Inglaterra com idade menor que 25 anos
#5. Média do número de gols da seleção da Italia
#6. Contar número de clubes diferentes de jogadores da Alemanha

In [81]:
# Adicionar arquivo ao contexto
url = 'https://raw.githubusercontent.com/sergiocalazans/big-data-pyspark/refs/heads/main/euro2024_players.csv'
sc.addFile(url)

In [82]:
# Carregar arquivo csv em um rdd utilizando SparkFiles para baixar a URL
rdd = sc.textFile(SparkFiles.get('euro2024_players.csv'))
rdd.take(5)

['Name,Position,Age,Club,Height,Foot,Caps,Goals,MarketValue,Country',
 'Marc-André ter Stegen,Goalkeeper,32,FC Barcelona,187,right,40,0,28000000,Germany',
 'Manuel Neuer,Goalkeeper,38,Bayern Munich,193,right,119,0,4000000,Germany',
 'Oliver Baumann,Goalkeeper,34,TSG 1899 Hoffenheim,187,right,0,0,3000000,Germany',
 'Nico Schlotterbeck,Centre-Back,24,Borussia Dortmund,191,left,12,0,40000000,Germany']

In [83]:
#retirar cabeçalho
header = rdd.first()
print(header)
rdd = rdd.\
  filter(lambda x: x!=header)
rdd.take(2)

Name,Position,Age,Club,Height,Foot,Caps,Goals,MarketValue,Country


['Marc-André ter Stegen,Goalkeeper,32,FC Barcelona,187,right,40,0,28000000,Germany',
 'Manuel Neuer,Goalkeeper,38,Bayern Munich,193,right,119,0,4000000,Germany']

In [84]:
#1. transformar o nome dos jogadores em letra maiúscula
rdd.\
  map(lambda x: x.split(',')[0].upper()).\
  take(2)

['MARC-ANDRÉ TER STEGEN', 'MANUEL NEUER']

In [85]:
#2. Exibir todos os jogadores da Alemanha
rdd.\
  filter(lambda x: x.split(',')[-1]=='Germany').\
  map(lambda x: (x.split(',')[0], x.split(',')[2])).\
  take(2)

[('Marc-André ter Stegen', '32'), ('Manuel Neuer', '38')]

In [86]:
#3. Contar número de jogadores da Inglaterra com idade menor que 25 anos
rdd.\
  filter(lambda x: x.split(',')[-1]=='England').\
  filter(lambda x: int(x.split(',')[2])<25).\
  count()

9

In [87]:
#4. Listar jogadores da Inglaterra com idade menor que 25 anos
rdd.\
  filter(lambda x: x.split(',')[-1]=='England').\
  filter(lambda x: int(x.split(',')[2])<25).\
  collect()

['Marc Guéhi,Centre-Back,23,Crystal Palace,182,right,11,0,38000000,England',
 'Conor Gallagher,Central Midfield,24,Chelsea FC,182,right,13,0,50000000,England',
 'Kobbie Mainoo,Central Midfield,19,Manchester United,175,right,3,0,50000000,England',
 'Adam Wharton,Central Midfield,20,Crystal Palace,182,left,1,0,30000000,England',
 'Jude Bellingham,Attacking Midfield,20,Real Madrid,186,right,29,3,180000000,England',
 'Cole Palmer,Attacking Midfield,22,Chelsea FC,189,left,4,1,80000000,England',
 'Anthony Gordon,Left Winger,23,Newcastle United,182,right,3,0,60000000,England',
 'Phil Foden,Right Winger,24,Manchester City,171,left,34,4,150000000,England',
 'Bukayo Saka,Right Winger,22,Arsenal FC,178,left,33,11,140000000,England']

In [88]:
#5. Média do número de gols da seleção da Italia
italia =rdd.\
  filter(lambda x: x.split(',')[-1]=='Italy').\
  map(lambda x: int(x.split(',')[7]))

#italia.collect()
italia.reduce(lambda x,y: x+y)/italia.count()

2.269230769230769

In [89]:
#6. Contar número de clubes diferentes de jogadores da Alemanha
rdd.\
  filter(lambda x: x.split(',')[-1] == 'Germany').\
  map(lambda x: x.split(',')[3]).\
  distinct().\
  count()

11

In [90]:
###############################################################
###############################################################
## Atividade
#1. Carregar csv no rdd
#2. Contagem do número de elementos do rdd
#3. Contagem do número de cidades diferentes
#4. Média de Rainfall pela cidade de Albury
#5. Máxima WindGustSpeed
#6. Contagem do número de dados por cidades
#7. Contar número de linhas com valor yes para rainToday
#8. Contar número de linhas com valor no para rainTomorrow
#9. Média de Temp3pm para dias que chove hoje
#10. Valor máximo e mínimo para Temp9am para dias em que chove amanhã

#Observação, tratar dado faltando

In [104]:
#1. Carregar csv no rdd
url_weather = 'https://raw.githubusercontent.com/sergiocalazans/big-data-pyspark/refs/heads/main/weather_aus.csv'
sc.addFile(url_weather)
rdd_weather = sc.textFile(SparkFiles.get('weather_aus.csv'))
rdd_weather.take(2)

['Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow',
 '2008-12-01,Albury,13.4,22.9,0.6,NA,NA,W,44,W,WNW,20,24,71,22,1007.7,1007.1,8,NA,16.9,21.8,No,No']

In [105]:
header_weather = rdd_weather.first()
rdd_weather = rdd_weather.\
  filter(lambda x: x!=header_weather)
rdd_weather.take(2)

['2008-12-01,Albury,13.4,22.9,0.6,NA,NA,W,44,W,WNW,20,24,71,22,1007.7,1007.1,8,NA,16.9,21.8,No,No',
 '2008-12-02,Albury,7.4,25.1,0,NA,NA,WNW,44,NNW,WSW,4,22,44,25,1010.6,1007.8,NA,NA,17.2,24.3,No,No']

In [106]:
#2. Contagem do número de elementos do rdd
rdd_weather.\
  count()

145460

In [107]:
#3. Contagem do número de cidades diferentes
rdd_weather.\
  map(lambda x: x.split(',')[1]).\
  distinct().\
  count()

49

In [108]:
#4. Média de Rainfall pela cidade de Albury
rainfall = rdd_weather.\
  filter(lambda x: x.split(',')[1] == 'Albury').\
  map(lambda x: float(x.split(',')[4] != 'NA'))

rainfall.reduce(lambda x,y: x+y)/rainfall.count()

0.9904605263157895

In [128]:
#4. Média de Rainfall pela cidade de Albury
rdd_weather.\
  filter(lambda x: x.split(',')[1] == 'Albury').\
  filter(lambda x: x.split(',')[4] != 'NA').\
  map(lambda x: float(x.split(',')[4])).\
  mean()

1.9141149119893657

In [112]:
#5. Máxima WindGustSpeed
rdd_weather.\
  filter(lambda x: x.split(',')[8] != 'NA').\
  map(lambda x: float(x.split(',')[8])).\
  max()

135.0

In [114]:
#6. Contagem do número de dados por cidades

rdd_weather.\
  filter(lambda x: x.split(',')[1] != 'NA').\
  map(lambda x: (x.split(',')[1])).\
  countByValue()

defaultdict(int,
            {'Albury': 3040,
             'BadgerysCreek': 3009,
             'Cobar': 3009,
             'CoffsHarbour': 3009,
             'Moree': 3009,
             'Newcastle': 3039,
             'NorahHead': 3004,
             'NorfolkIsland': 3009,
             'Penrith': 3039,
             'Richmond': 3009,
             'Sydney': 3344,
             'SydneyAirport': 3009,
             'WaggaWagga': 3009,
             'Williamtown': 3009,
             'Wollongong': 3040,
             'Canberra': 3436,
             'Tuggeranong': 3039,
             'MountGinini': 3040,
             'Ballarat': 3040,
             'Bendigo': 3040,
             'Sale': 3009,
             'MelbourneAirport': 3009,
             'Melbourne': 3193,
             'Mildura': 3009,
             'Nhil': 1578,
             'Portland': 3009,
             'Watsonia': 3009,
             'Dartmoor': 3009,
             'Brisbane': 3193,
             'Cairns': 3040,
             'GoldCoast': 3040,
 

In [124]:
#7. Contar número de linhas com valor yes para rainToday
rainToday_yes = rdd_weather.\
  filter(lambda x: x.split(',')[-2] != 'NA').\
  filter(lambda x: (x.split(',')[-2]).lower() == 'yes')

print(rainToday_yes.count())

31880


In [117]:
#8. Contar número de linhas com valor no para rainTomorrow
rdd_weather.\
  filter(lambda x: x.split(',')[-1] != 'NA').\
  filter(lambda x: (x.split(',')[-1]).lower() == 'no').\
  count()

110316

In [129]:
#9. Média de Temp3pm para dias que chove hoje
# rainToday_yes já possui o filtro onde rainToday == 'yes'
rainToday_Temp3pm = rainToday_yes.\
  filter(lambda x: x.split(',')[-3] != 'NA').\
  map(lambda x: float(x.split(',')[-3]))

print(rainToday_Temp3pm.mean())

18.64479427434353


In [130]:
#10. Valor máximo e mínimo para Temp9am para dias em que chove amanhã
# Temp9am é o índice 19 (ou -4 na ordem reversa)
rainTomorrow_yes = rdd_weather.\
  filter(lambda x: x.split(',')[-1] != 'NA').\
  filter(lambda x: (x.split(',')[-1]).lower() == 'yes')

rainTomorrow_Temp9am = rainTomorrow_yes.\
  filter(lambda x: x.split(',')[-4] != 'NA').\
  map(lambda x: float(x.split(',')[-4]))

print("Máximo:", rainTomorrow_Temp9am.max())
print("Mínimo:", rainTomorrow_Temp9am.min())

Máximo: 37.5
Mínimo: -5.5
